In [1]:
print("Hola mundo — F1, grupo 5")

Hola mundo — F1, grupo 5


# F1 — Definición del proyecto y entorno reproducible
## Ofertas en licitaciones públicas del sector Salud de marzo de 2026
**Curso:** MCDIA500, Programación para la Ciencia de Datos. **Grupo:** 5.
**Integrantes:** Víctor Bravo Barrera, Nayadeth Garrido y Alexander Sepulveda.

Esta fase define el problema, los objetivos y las condiciones técnicas para estudiar las ofertas del archivo seleccionado. Implementa la verificación del entorno y la lectura inicial trazable. La limpieza y el cálculo de resultados analíticos se desarrollarán en F2 y fases posteriores; las comprobaciones de este notebook no constituyen conclusiones sobre adjudicación.

## 1. Contexto y problemática
El proyecto utiliza el archivo de licitaciones del sector Salud de marzo de 2026, cuya procedencia está documentada en [data/README.md](../data/README.md) y remite al portal de datos abiertos de ChileCompra (ChileCompra, s. f.). Los registros relacionan procesos de compra, ítems y ofertas de proveedores.

La necesidad analítica es describir si la proporción de ofertas registradas como ganadoras varía entre tipos de licitación y tamaños de proveedor. El volumen de registros y la combinación de variables categóricas, temporales y monetarias requieren lectura programática, verificaciones explícitas y un flujo reproducible. Una comparación sin revisar el significado de las filas, los faltantes y las categorías podría producir resultados engañosos.

La **unidad de análisis de trabajo** es una oferta asociada a un ítem de una licitación. `NroLicitacion` identifica el proceso y puede repetirse legítimamente. La clave que distingue cada observación debe contrastarse con el diccionario y los datos en F2; no se presupone que el identificador de licitación sea único por fila.

## 2. Preguntas de análisis
**Pregunta central:** ¿cómo varía la proporción de ofertas ganadoras según el tipo de licitación y el tamaño del proveedor en el sector Salud durante marzo de 2026?

1. ¿Cómo se distribuyen los registros por `TipoLicitacion` y `TamanoProveedor`?
2. ¿Qué proporción de ofertas presenta `ResultadoOferta` igual a `Ganadora` en cada grupo?
3. ¿Qué problemas de cobertura, faltantes o clasificación limitan la interpretación de esas comparaciones?

La proporción prevista será el número de ofertas con resultado `Ganadora` dividido por las ofertas con resultado válido (`Ganadora` o `Perdedora`) del grupo. Se informarán también denominadores y resultados faltantes o no reconocidos; no se convertirán automáticamente en pérdidas. Un grupo sin resultados válidos tendrá proporción no definida. La homogeneización de etiquetas y su comprobación corresponde a F2.

## 3. Objetivos
### Objetivo general
Analizar descriptivamente las diferencias en la proporción de ofertas ganadoras según tipo de licitación y tamaño del proveedor en el archivo de licitaciones del sector Salud de marzo de 2026, mediante un flujo reproducible que permita verificar la calidad de los datos y los resultados.

### Objetivos específicos
1. Delimitar la pregunta, la unidad de análisis, las variables y los supuestos, y configurar un entorno reproducible con trazabilidad del archivo de entrada (F1).
2. Caracterizar la estructura y calidad de los datos, documentar los roles de las variables y preparar un conjunto validado mediante reglas justificadas (F2).
3. Calcular y visualizar distribuciones y proporciones por tipo de licitación y tamaño del proveedor, comunicando los denominadores y las limitaciones (proyección F3).
4. Integrar resultados, evidencias de ejecución y conclusiones en una entrega reproducible y coherente con los objetivos (proyección F4).

La proyección F3–F4 es una planificación del equipo y deberá ajustarse a las pautas específicas de esas fases cuando estén disponibles.

## 4. Alcance, restricciones y supuestos
- Se trabaja con un solo archivo correspondiente al reporte seleccionado de marzo de 2026, sector Salud. La cobertura declarada se contrastará con `Sector` y las fechas en F2; no se asume que toda fecha administrativa pertenezca a marzo.
- El enfoque es descriptivo. Las asociaciones no demuestran causalidad ni permiten evaluar por sí solas la calidad del proveedor o la eficiencia de una compra.
- Las filas pueden compartir licitación y proveedor; no son necesariamente observaciones independientes. La proporción por oferta no equivale a la proporción de proveedores o licitaciones adjudicadas.
- La etiqueta `ResultadoOferta` representa el resultado registrado en el archivo, no una verificación independiente de contratos o pagos.
- Se conserva el CSV original. No se imputan, eliminan ni normalizan datos en F1. Los faltantes y categorías desconocidas se tratarán explícitamente en F2.
- Los montos no forman parte de la comparación principal. Cualquier extensión monetaria requerirá comprobar moneda, unidad y granularidad antes de sumar o comparar.
- No se generalizan resultados a otros meses ni a todo el sistema de compras públicas. Los tamaños de grupo se informarán para evitar interpretar porcentajes sin contexto.
- Quedan pendientes el diccionario oficial, el contraste de roles del equipo y la confirmación de la correspondencia con el caso y mapa conceptual del curso.

## 5. Variables previstas y procedencia
| Variable | Rol propuesto por significado | Uso y precaución |
| --- | --- | --- |
| NroLicitacion | Identificador del proceso | Agrupación y trazabilidad; no clave única de fila |
| TipoLicitacion | Categórica nominal | Comparación entre tipos; revisar categorías |
| TamanoProveedor | Categórica | Comparación por tamaño; no imponer orden hasta revisar definiciones |
| ResultadoOferta | Categórica binaria esperada | Resultado registrado; validar etiquetas y faltantes |
| Sector | Categórica nominal | Comprobación del alcance sectorial |
| FechaPublicacion | Temporal esperada | Contraste de cobertura; conversión en F2 |

Esta selección es una definición inicial, no un diccionario completo de las 74 columnas ni el resultado de un clasificador automático. El archivo se lee con separador `;` y codificación `latin-1`, parámetros explícitos de `pandas.read_csv` (The pandas development team, s. f.). La inferencia de tipos durante la lectura no sustituye la validación semántica.

## 6. Herramientas y entorno
Python organiza el flujo mediante funciones; pandas permite leer el CSV; NumPy se comprueba con una operación sintética; Jupyter integra narrativa, código y salidas. Git registra la evolución del proyecto. La prueba de NumPy usa `arange` para construir una secuencia entera verificable (NumPy Developers, s. f.).

Las instrucciones de instalación están en el [README](../README.md) y las dependencias principales en `requirements.txt`. Las versiones utilizadas y la versión efectiva de Python se registran abajo. La referencia previa a Python 3.14.7 no demuestra compatibilidad: esta revisión se verifica en Python 3.12 y cualquier otro entorno requiere su propia ejecución completa.

In [1]:
from pathlib import Path
import sys
import platform

# La estructura identifica la raíz incluso al abrir el notebook desde F1.
raiz = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "proyecto.py").is_file() and (p / "F1").is_dir()), None)
if raiz is None:
    raise FileNotFoundError("Abrir el notebook desde la raíz del repositorio o F1.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
from src.proyecto import versiones_entorno, sha256_archivo, leer_datos_f1

entorno = {"python": platform.python_version(), "sistema": platform.platform(),
           "entorno_virtual": sys.prefix != sys.base_prefix,
           "dependencias": versiones_entorno()}
display(entorno)
muestra = pd.DataFrame({"valor": np.arange(3)})
assert muestra["valor"].sum() == 3
print("OK: integración mínima de NumPy y pandas.")

{'python': '3.14.7',
 'sistema': 'Windows-11-10.0.26200-SP0',
 'entorno_virtual': False,
 'dependencias': {'numpy': '2.5.3',
  'pandas': '2.3.3',
  'jupyterlab': '4.6.3',
  'ipykernel': '7.3.0',
  'nbformat': '5.11.1',
  'nbconvert': '7.17.1',
  'nbclient': '0.11.0'}}

OK: integración mínima de NumPy y pandas.


## 7. Lectura inicial y trazabilidad del dataset
Se verifica la huella SHA-256 documentada antes de leer el archivo. Las dimensiones esperadas describen esta versión concreta del CSV; un cambio exige revisar la procedencia y actualizar la documentación, no ignorar la comprobación. La lectura conserva las filas y columnas y no genera datos procesados.

In [4]:
ruta_datos = raiz / "data" / "raw" / "licitaciones_salud_marzo_2026.csv"
columnas_requeridas = ["NroLicitacion", "TipoLicitacion", "TamanoProveedor",
                       "ResultadoOferta", "Sector", "FechaPublicacion"]
huella_esperada = "490d9209a10d387011d481b72b7891f26e997974ec2cf9dfc518aa4a08552232"
huella = sha256_archivo(ruta_datos)
assert huella == huella_esperada, "El CSV difiere de la versión documentada."
datos = leer_datos_f1(ruta_datos, columnas_requeridas)
assert datos.shape == (44226, 74), "Revisar dimensiones de la versión de entrada."
resumen_entrada = {"archivo": ruta_datos.name, "filas": len(datos),
                  "columnas": len(datos.columns), "sha256": huella,
                  "columnas_requeridas_presentes": True}
display(resumen_entrada)
display(pd.DataFrame({"variable": columnas_requeridas,
                      "tipo_inferido": [str(datos[c].dtype) for c in columnas_requeridas]}))
print("OK: entrada disponible, íntegra y con el esquema mínimo esperado.")

{'archivo': 'licitaciones_salud_marzo_2026.csv',
 'filas': 44226,
 'columnas': 74,
 'sha256': '490d9209a10d387011d481b72b7891f26e997974ec2cf9dfc518aa4a08552232',
 'columnas_requeridas_presentes': True}

,variable,tipo_inferido
0,NroLicitacion,object
1,TipoLicitacion,object
2,TamanoProveedor,object
3,ResultadoOferta,object
4,Sector,object
5,FechaPublicacion,object


OK: entrada disponible, íntegra y con el esquema mínimo esperado.


## 8. Verificación del código inicial
Las pruebas siguientes comprueban el contrato de lectura con entradas pequeñas: caso normal, ausencia de archivo, ausencia de una columna y archivo con encabezado pero sin registros. No son pruebas del futuro pipeline F2. Los archivos de prueba se crean en una carpeta temporal que se elimina al terminar.

In [7]:
from tempfile import TemporaryDirectory

pruebas = []
with TemporaryDirectory() as temporal:
    carpeta = Path(temporal)
    normal = carpeta / "normal.csv"
    normal.write_text("TipoLicitacion;ResultadoOferta\nLE;Ganadora\n", encoding="latin-1")
    resultado = leer_datos_f1(normal, ["TipoLicitacion", "ResultadoOferta"])
    assert resultado.shape == (1, 2) and resultado.loc[0, "ResultadoOferta"] == "Ganadora"
    pruebas.append({"caso": "Lectura normal", "resultado": "OK"})

    vacio = carpeta / "vacio.csv"
    vacio.write_text("TipoLicitacion;ResultadoOferta\n", encoding="latin-1")
    casos = [
        ("Archivo ausente", carpeta / "ausente.csv", ["TipoLicitacion"], FileNotFoundError),
        ("Columna ausente", normal, ["TamanoProveedor"], ValueError),
        ("Sin registros", vacio, ["TipoLicitacion"], ValueError),
    ]
    for nombre, ruta, columnas, excepcion in casos:
        try:
            leer_datos_f1(ruta, columnas)
        except excepcion as error:
            pruebas.append({"caso": nombre, "resultado": "OK", "detalle": str(error)})
        else:
            raise AssertionError(f"No se detectó el caso: {nombre}")
display(pd.DataFrame(pruebas))
assert sha256_archivo(ruta_datos) == huella, "El original fue modificado."
print("OK: cuatro pruebas y conservación del archivo original.")

,caso,resultado,detalle
0,Lectura normal,OK,NaN
1,Archivo ausente,OK,No existe un archivo de datos en: C:\Users\vbr...
2,Columna ausente,OK,Faltan columnas requeridas: TamanoProveedor
3,Sin registros,OK,El dataset no contiene registros.


OK: cuatro pruebas y conservación del archivo original.


## 9. Decisiones y articulación con las fases
| Decisión | Motivo | Evidencia o fase |
| --- | --- | --- |
| Acotar a tipo de licitación y tamaño de proveedor | Dar una pregunta medible al proyecto | Secciones 2 y 3 |
| Mantener granularidad de oferta por ítem | Evitar confundir filas con licitaciones únicas | Secciones 1 y 4; contraste en F2 |
| Leer sin limpiar y comprobar SHA-256 | Preservar la versión de origen | Secciones 7 y 8 |
| Usar funciones, sin añadir clases | Las operaciones iniciales no requieren mantener estado | src/proyecto.py |
| Separar entorno y datos originales de resultados | Facilitar reproducción y auditoría | README, data/raw y evidencias |
| Posponer comparaciones monetarias | Exigen resolver monedas y granularidad | Restricción de alcance |

F1 entrega definición y entorno; F2 materializará exploración, limpieza, transformación y validación del dataset; F3 y F4 quedan proyectadas para análisis y comunicación, sujetos a sus pautas.

### Vinculación con el mapa conceptual
**El mapa conceptual original no está disponible en el repositorio revisado.** La tabla siguiente permite preparar el contraste, pero no certifica correspondencia con un mapa no revisado.

| Componente técnico a contrastar | Implementación disponible | Pendiente |
| --- | --- | --- |
| Problema y objetivos | Secciones 1–4 de F1 | Contrastar con el mapa del equipo |
| Python, NumPy, pandas y Jupyter | Secciones 6–8, requirements y src | Relacionar con nodos del mapa |
| Entorno y trazabilidad | README, SHA-256 y salidas ejecutadas | Incorporar referencia del commit final |
| Preparación de datos | Planificación de F2 | Implementar y justificar reglas en F2 |
| Análisis y comunicación | Objetivos proyectados F3–F4 | Ajustar a pautas e informe |

El historial Git permite identificar las contribuciones; no se atribuyen a integrantes cambios que aún no tienen commit. Este notebook deberá incorporarse al informe integrado con referencias a sus evidencias cuando se desarrolle ese documento.

## 10. Referencias y pendientes documentales
ChileCompra. (s. f.). *Datos abiertos: Descargas*. Recuperado el 12 de septiembre de 2026, de https://datos-abiertos.chilecompra.cl/descargas

The pandas development team. (s. f.). *pandas.read_csv*. pandas documentation. Recuperado el 12 de septiembre de 2026, de https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html

NumPy Developers. (s. f.). *numpy.arange*. NumPy documentation. Recuperado el 12 de septiembre de 2026, de https://numpy.org/doc/stable/reference/generated/numpy.arange.html

Las dos documentaciones oficiales sustentan las operaciones iniciales descritas en las secciones 5 y 6. **La bibliografía de la entrega aún no está completa:** faltan dos recursos docentes y una fuente académica complementaria pertinente y reciente, con citas en el desarrollo. La fuente del dataset no sustituye esos requisitos.

Para cerrar documentalmente F1 se requiere incorporar el mapa del equipo, el diccionario oficial o su contraste, las fuentes faltantes y la referencia al commit de esta revisión. No se presentan como entregados ni verificados.